# 15 — Milestone 2: parse every paper on Drive

Owner: Member 1. **The notebook to open on Google Colab (CPU runtime) for Milestone 2.** It loads `00_config`,
`02_download` (only for its run lock and its receipts) and `13_dedup_export` (which loads `10`–`12` when they
exist, see the contract in `13`), then:

1. **Papers**: the IDs of `state/discovery_manifest.json` that exist on arXiv, not the folders of `data/raw/`
   (it also holds `.staging`, the downloader's work folder). `TEST_PAPERS` takes that many, spread over the
   four ranges, plus the hard cases (several roots, `\begin{document}` in an included file, no section, many
   versions, `\chapter`, missing `.bbl`).
2. **Parse in parallel**: `WORKERS` processes run `parse_paper` and write `data/raw/<id>/parsed.json`; the
   main process appends `logs/m2_parse_status.csv` (one row per version) and the resume index
   `state/m2/done.jsonl`. A paper that fails is logged as `error` and does not stop the run.
3. **Tables**: the `parsed.json` of every paper → `data/m2/elements.parquet`, `hierarchy.parquet`,
   `bibtex.parquet`, written 500 papers at a time with `pyarrow` (never the whole corpus in RAM), with the
   invariant checks.
4. **Statistics and benchmark**: `logs/m2_stats.json` (for the report) and `logs/m2_benchmark.json` over
   every session.

**Resume.** After a disconnect, run all cells again: the papers in `state/m2/done.jsonl` with the current
`PARSER_VERSION` are skipped (one file read instead of 20,000 `parsed.json` on Drive). Each session saves its
time and RAM to `logs/m2_sessions/` every 5 minutes, so a session that Colab kills still counts.

**Before the full run**: set `TEST_PAPERS = None` and `FRESH_RUN = True` once (the test's logs and sessions
are deleted, so the benchmark covers the full run only), then `FRESH_RUN = False` for the next sessions.

## 0. Setup

`WORKERS` processes parse in parallel (`multiprocessing`, `fork` on Colab's Linux, so the functions defined
in the notebooks work in the workers). Reading from Drive is the bottleneck, not the CPU, so more workers than
Colab's 2 cores still help; 4–8 is the range to try. Without `fork` (Windows) the papers are parsed one by
one. `M2_ROOT` points to another project folder with `data/raw`, `state` and `logs` (e.g. a local copy or
`test_runs/1000_versions`); another notebook can set `M2_OVERRIDES = {...}` before `%run -i` to change any
value of this cell.

In [ ]:
# On Colab: clone or update the branch, install packages and work from src/ (the notebooks use relative paths).
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB:
    # Every run: the notebooks loaded with %run must be the latest commit of the branch.
    REPO = Path("/content/intro-ds-arxiv-project")
    BRANCH = "Dedup"   # Milestone 2 branch, "VA" once merged
    URL = "https://github.com/thinhelzan842/intro-ds-arxiv-project.git"
    if not (REPO / ".git").is_dir():
        subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", URL, str(REPO)], check=True)
    else:   # explicit refspec: also works in a clone of another single branch (e.g. VA, cloned by 05)
        subprocess.run(["git", "-C", str(REPO), "fetch", "origin",
                        f"+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"], check=True)
        subprocess.run(["git", "-C", str(REPO), "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "src/requirements.txt")], check=True)
    os.chdir(REPO / "src")
    print("Code in use:", subprocess.run(["git", "-C", str(REPO), "log", "--oneline", "-1"],
                                         capture_output=True, text=True).stdout.strip())

RUN_TEST = False          # True also runs the offline tests of 02 and 13
TEST_PAPERS = 100         # None: every paper; N: N papers spread over the four ranges, plus HARD_CASES
HARD_CASES = ("2609-32839", "2609-32858", "2603-21039", "2609-33000", "2605-22101", "2609-33059",
              "2603-22481", "2603-20714", "2603-23413")
WORKERS = 6               # parsing processes (4-8: Drive reads are the bottleneck, not the 2 CPU cores)
FRESH_RUN = False         # True: delete the M2 logs, sessions, index and tables first (parsed.json is overwritten)
MERGE_TABLES = True       # build the three Parquet tables at the end
M2_ROOT = None            # another folder with data/raw, state, logs (default: PROJECT_ROOT)
FLUSH_DRIVE_AT_END = True  # flush and unmount Google Drive at the end, so every file is saved

globals().update(globals().get("M2_OVERRIDES", {}))

In [ ]:
get_ipython().run_line_magic("run", "-i ./00_config.ipynb")
if M2_ROOT:
    # Another project folder: move every path before 02 and 13 derive theirs from them.
    DATA_DIR, LOG_DIR, STATE_DIR = Path(M2_ROOT) / "data" / "raw", Path(M2_ROOT) / "logs", Path(M2_ROOT) / "state"
    PAPER_LOG = LOG_DIR / "papers.csv"
    for _folder in (LOG_DIR, STATE_DIR):
        _folder.mkdir(parents=True, exist_ok=True)
    print(f"M2_ROOT: {M2_ROOT}")

for notebook in ("02_download", "13_dedup_export"):
    get_ipython().run_line_magic("run", f"-i ./{notebook}.ipynb")

import multiprocessing
import platform
import shutil
import statistics
import tempfile
import threading
import uuid
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import psutil
import pyarrow as pa
import pyarrow.parquet as pq

M2_DATA = DATA_DIR.parent / "m2"                    # data/m2/*.parquet
M2_STATE = STATE_DIR / "m2"
DONE_INDEX = M2_STATE / "done.jsonl"                # resume index: one line per parsed paper
STATUS_LOG = LOG_DIR / "m2_parse_status.csv"        # one row per version
STATUS_FIELDS = ["arxiv_id", "version", "status", "bib_source", "n_nodes", "seconds", "error"]
M2_SESSIONS = LOG_DIR / "m2_sessions"               # one record per Colab session
M2_SESSIONS_LOG = LOG_DIR / "m2_benchmark_sessions.jsonl"
M2_RAM_LOG = LOG_DIR / "m2_ram_timeseries.csv"
M2_BENCHMARK = LOG_DIR / "m2_benchmark.json"
M2_STATS = LOG_DIR / "m2_stats.json"
MB = 1024 * 1024

if FRESH_RUN:
    for path in (STATUS_LOG, M2_SESSIONS_LOG, M2_RAM_LOG, M2_BENCHMARK, M2_STATS):
        path.unlink(missing_ok=True)
    for folder in (M2_SESSIONS, M2_STATE, M2_DATA):
        shutil.rmtree(folder, ignore_errors=True)
    print("FRESH_RUN: M2 logs, sessions, index and tables deleted")
M2_STATE.mkdir(parents=True, exist_ok=True)
print(f"Parser {PARSER_VERSION} | workers {WORKERS} | data {DATA_DIR}")

## 1. Papers to parse

The IDs come from the discovery manifest (papers with at least one version); without a manifest, from the
folders of `data/raw/` whose name is an ID. The folders that are not an assigned paper are printed (on Drive:
`.staging`, the downloader's work folder, which makes 20,001 folders for 20,000 IDs). `versions` here is the
count of `metadata.json` / the manifest, never the `vN/` folders.

In [ ]:
ID_FOLDER = re.compile(r"^\d{4}-\d{4,5}$")


def papers_to_parse():
    """{folder_id: number of versions} of every paper, in manifest order."""
    manifest = read_json(STATE_DIR / "discovery_manifest.json")
    with os.scandir(DATA_DIR) as entries:
        folders = {entry.name for entry in entries if entry.is_dir()}
    if manifest:
        records = manifest["papers"] if isinstance(manifest, dict) else manifest
        papers = {to_folder_id(p["id"]): int(p["versions"]) for p in records if p.get("versions", 0) > 0}
    else:
        papers = {}
        for name in sorted(folders):
            if ID_FOLDER.match(name):
                meta = read_json(DATA_DIR / name / "metadata.json")
                if meta and meta.get("versions"):
                    papers[name] = int(meta["versions"])
    extra = sorted(folders - set(papers))
    missing = [f for f in papers if f not in folders]
    print(f"{len(papers):,} papers ({sum(papers.values()):,} versions) | {len(folders):,} folders in data/raw"
          + (f" | not a paper: {', '.join(extra[:10])}" + (" ..." if len(extra) > 10 else "") if extra else "")
          + (f" | {len(missing):,} papers without a folder" if missing else ""))
    return papers


def read_done_index(path=DONE_INDEX):
    """{folder_id: summary} of the papers parsed with the current PARSER_VERSION (last line wins)."""
    done = {}
    try:
        with open(path, encoding="utf-8") as stream:
            for line in stream:
                try:
                    summary = json.loads(line)
                except ValueError:
                    continue   # a line torn by a killed session
                if summary.get("parser_version") == PARSER_VERSION:
                    done[summary["folder_id"]] = summary
                else:
                    done.pop(summary.get("folder_id"), None)
    except FileNotFoundError:
        pass
    return done


ALL_PAPERS = papers_to_parse()
if TEST_PAPERS:
    _ids = list(ALL_PAPERS)
    _step = max(1, len(_ids) // TEST_PAPERS)
    _chosen = dict.fromkeys(_ids[::_step][:TEST_PAPERS] + [f for f in HARD_CASES if f in ALL_PAPERS])
    PAPERS = {f: ALL_PAPERS[f] for f in _chosen}
    print(f"TEST_PAPERS: {len(PAPERS):,} papers ({sum(f in ALL_PAPERS for f in HARD_CASES)} hard cases)")
else:
    PAPERS = ALL_PAPERS

done = read_done_index()
TODO = [f for f in PAPERS if f not in done]
print(f"{len(PAPERS) - len(TODO):,} already parsed with this parser version, {len(TODO):,} to parse")

## 2. Parse in parallel, with the benchmark running

`parse_one` runs in a worker process: `parse_paper`, then `parsed.json` next to the paper's `metadata.json`
(`write_json`: temporary file + rename; compact JSON, 25% smaller than indented, still about 1.5 × the
paper's `.tex` because JSON doubles every backslash), and a small summary back to the main process (counts,
no rows). The
main process alone writes the logs, so no two processes append to one file:

- `logs/m2_parse_status.csv`: `arxiv_id, version, status, bib_source, n_nodes, seconds, error`, one row per
  version. For an `empty_source` version the download receipts say why: `pdf_only` (arXiv has no TeX
  source), `not_found` (HTTP 404, e.g. a withdrawn version), `files_missing` (the receipt says the version
  was downloaded but its `vN/` folder is not there) or `not_downloaded`.
- `state/m2/done.jsonl`: the summary, written after `parsed.json`; a paper whose parsing failed is not in it
  and is parsed again next time.

**Benchmark.** RAM is sampled every 2 s as the RSS of this process **plus its worker processes**
(`psutil`, `children(recursive=True)`); the pool is started before the sampling thread. The session record
(`logs/m2_sessions/<id>.json`) is saved every 5 minutes and the samples appended to
`logs/m2_ram_timeseries.csv`, as in `05_run_pipeline`. A progress bar shows papers, versions, errors and the
ETA; a text line every 500 papers stays in the output after a browser reconnect.

In [ ]:
try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None
if ON_COLAB and tqdm is not None:
    from tqdm.std import tqdm   # Colab: text bars, which survive a browser reconnect


def parse_one(folder_id):
    """Worker: parse one paper and write its parsed.json; returns a summary without the rows."""
    pdir = DATA_DIR / folder_id
    started = time.perf_counter()
    try:
        result = parse_paper(pdir)
        write_json(pdir / "parsed.json", to_parsed_json(result), indent=None)   # compact JSON
        return {
            "folder_id": folder_id, "arxiv_id": result["arxiv_id"], "parser_version": result["parser_version"],
            "ok": True, "versions": result["versions"],
            "nodes_by_kind": result["counts"]["nodes_by_kind"],
            "elements_by_kind": dict(Counter(row["kind"] for row in result["elements"])),
            "bib_before": result["counts"]["bib_entries_before"], "bib_after": len(result["bibtex"]),
            "rows": {table: len(result[table]) for table in ("elements", "hierarchy", "bibtex")},
            "bytes": (pdir / "parsed.json").stat().st_size,
            "problems": check_paper(result, len(result["versions"])),
            "seconds": round(time.perf_counter() - started, 3),
        }
    except Exception as err:   # the whole paper failed (e.g. no metadata.json): every version is an error
        try:
            n_versions = int((read_json(pdir / "metadata.json") or {}).get("versions") or 0)
        except Exception:
            n_versions = 0
        n_versions = n_versions or PAPERS.get(folder_id, 1)
        return {"folder_id": folder_id, "arxiv_id": folder_id.replace("-", ".", 1), "ok": False,
                "versions": [{"version": v, "status": "error", "bib_source": "", "n_nodes": 0, "seconds": 0.0,
                              "error": _error_text(err)} for v in range(1, n_versions + 1)],
                "seconds": round(time.perf_counter() - started, 3)}


def empty_reasons():
    """{(arxiv_id, version): reason} for versions without source, from the download receipts and errors."""
    # receipt "ok" but no vN/ folder: the files never reached Drive (e.g. a session killed before Drive synced)
    reasons = {key: "pdf_only" if receipt.get("status") == "empty_source" else "files_missing"
               for key, receipt in _read_receipt_index(DOWNLOAD_STATE).items()}
    try:
        with open(DOWNLOAD_STATE / "errors.jsonl", encoding="utf-8") as stream:
            for line in stream:
                try:
                    row = json.loads(line)
                except ValueError:
                    continue
                key = (row.get("arxiv_id"), row.get("version"))
                if key not in reasons and (row.get("error_kind") == "NOT_FOUND" or "HTTP 404" in row.get("error", "")):
                    reasons[key] = "not_found"
    except FileNotFoundError:
        pass
    return reasons


def cpu_info():
    model = platform.processor() or ""
    try:
        with open("/proc/cpuinfo", encoding="utf-8") as stream:
            model = next((l.split(":", 1)[1].strip() for l in stream if l.startswith("model name")), model)
    except OSError:
        pass
    return f"{model} x {os.cpu_count()}".strip()


class M2Monitor:
    """RSS of this process and of all its children, every `interval` seconds, in a background thread."""

    def __init__(self, interval=2.0):
        self.interval, self.samples = interval, []   # samples: (unix time, MB)
        self.start_time = self.end_time = None
        self._process = psutil.Process()
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._loop, daemon=True)

    def rss_mb(self):
        total = self._process.memory_info().rss
        for child in self._process.children(recursive=True):
            try:
                total += child.memory_info().rss
            except psutil.Error:
                pass   # a worker that just ended
        return total / MB

    def _loop(self):
        while not self._stop.is_set():
            try:
                self.samples.append((time.time(), self.rss_mb()))
            except Exception:
                pass
            self._stop.wait(self.interval)

    def start(self):
        self.start_time = time.time()
        self._thread.start()
        return self

    def stop(self):
        if self.end_time is None:
            self.end_time = time.time()
        self._stop.set()
        self._thread.join(timeout=5)


class M2Session:
    """This session in logs/m2_sessions/<id>.json, saved every `every` seconds (complete: false until the end)."""

    def __init__(self, monitor, every=300):
        self.monitor, self.every = monitor, every
        started = datetime.fromtimestamp(monitor.start_time, tz=timezone.utc)
        self.id = started.strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:6]
        self.path = M2_SESSIONS / f"{self.id}.json"
        self.papers = self.versions = 0
        self.flushed = 0
        self._guard = threading.Lock()
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()

    def _loop(self):
        while not self._stop.wait(self.every):
            try:
                self.save()
            except Exception as err:   # a full Drive must not stop the parsing
                print(f"[benchmark] could not save this session: {err!r}")

    def save(self, final=None):
        with self._guard:
            monitor = self.monitor
            end = monitor.end_time or time.time()
            samples = list(monitor.samples)
            rams = [mb for _, mb in samples]
            record = {
                "session": self.id, "complete": final is not None,
                "start": to_iso(datetime.fromtimestamp(monitor.start_time, tz=timezone.utc)),
                "end": to_iso(datetime.fromtimestamp(end, tz=timezone.utc)),
                "wall_seconds": round(end - monitor.start_time, 3),
                "papers": self.papers, "versions": self.versions, "workers": WORKERS,
                "ram_peak_mb": round(max(rams, default=0.0), 2),
                "ram_avg_mb": round(sum(rams) / len(rams), 2) if rams else None,
                "ram_samples": len(rams), "parser_version": PARSER_VERSION,
            }
            record.update(final or {})
            M2_SESSIONS.mkdir(parents=True, exist_ok=True)
            write_json(self.path, record)
            new = samples[self.flushed:]
            if new:
                header = not M2_RAM_LOG.exists()
                with M2_RAM_LOG.open("a", newline="", encoding="utf-8") as stream:
                    writer = csv.writer(stream)
                    if header:
                        writer.writerow(["utc_time", "rss_mb"])
                    for moment, mb in new:
                        writer.writerow([to_iso(datetime.fromtimestamp(moment, tz=timezone.utc)), round(mb, 2)])
                self.flushed = len(samples)
            return record

    def finish(self, final):
        self._stop.set()
        self._thread.join(timeout=5)
        return self.save(final=final)


def run_parsing(todo):
    """Parse `todo` (folder IDs) with WORKERS processes; logs and index are written as results arrive."""
    reasons = empty_reasons()
    use_fork = WORKERS > 1 and "fork" in multiprocessing.get_all_start_methods()
    pool = multiprocessing.get_context("fork").Pool(WORKERS) if use_fork and todo else None   # before any thread
    lock = RunLock(M2_STATE / "parse.lock").acquire()
    monitor = M2Monitor().start()
    session = M2Session(monitor)
    counts = Counter()
    new_log = not STATUS_LOG.exists()
    bar = tqdm(total=len(todo), unit="paper", desc="Parsing", dynamic_ncols=True, mininterval=2) \
        if tqdm and todo else None
    started = time.perf_counter()
    try:
        with open(STATUS_LOG, "a", newline="", encoding="utf-8") as log, open(DONE_INDEX, "a", encoding="utf-8") as index:
            writer = csv.DictWriter(log, fieldnames=STATUS_FIELDS, extrasaction="ignore")
            if new_log:
                writer.writeheader()
            results = pool.imap_unordered(parse_one, todo) if pool else map(parse_one, todo)
            for n, summary in enumerate(results, 1):
                for record in summary["versions"]:
                    row = dict(record, arxiv_id=summary["arxiv_id"])
                    if row["status"] == "empty_source":
                        row["error"] = reasons.get((summary["arxiv_id"], row["version"]), "not_downloaded")
                    writer.writerow(row)
                    counts[row["status"]] += 1
                log.flush()
                if summary["ok"]:
                    index.write(json.dumps(summary, ensure_ascii=False) + "\n")
                    index.flush()
                else:
                    counts["paper_error"] += 1
                counts["problems"] += bool(summary.get("problems"))
                session.papers += 1
                session.versions += len(summary["versions"])
                if bar is not None:
                    bar.update(1)
                    bar.set_postfix_str(f"versions ok {counts['ok']:,} | empty {counts['empty_source']:,} | "
                                        f"no root {counts['no_root']:,} | errors {counts['error']:,}")
                if n % 500 == 0 or n == len(todo):
                    elapsed = time.perf_counter() - started
                    eta = elapsed / n * (len(todo) - n)
                    print(f"[{datetime.now().strftime('%H:%M:%S')}] {n:,}/{len(todo):,} papers | "
                          f"{elapsed / n:.2f} s/paper | ETA {eta / 60:,.1f} min | {dict(counts)}", flush=True)
    except KeyboardInterrupt:
        print("Interrupted: finished papers are saved; run all cells again to continue.")
    finally:
        if bar is not None:
            bar.close()
        if pool is not None:
            pool.terminate()
            pool.join()
        lock.release()
    return monitor, session, counts


monitor, session, run_counts = run_parsing(TODO)

## 3. The three tables

`merge_tables` reads the `parsed.json` of the papers of this run (with the current parser version), 500 papers
at a time with 16 threads (Drive reads), and appends each batch to the three Parquet files with
`pyarrow.parquet.ParquetWriter`, so memory holds one batch only. The files are written on the local disk
and copied to `data/m2/` at the end. Types follow the statement: `kind` and `entry_type` are
dictionary-encoded strings, `version` and `year` integers (`year` nullable), `authors` a list of strings,
`element_id`, `arxiv_id`, `parent_id` and `citation_key` non-null.

The same pass checks the invariants on every batch (a paper's rows are always in one batch): no duplicate
`element_id`; every `element_id` and `parent_id` of `hierarchy` is in `elements`; in each (paper, version)
the only parent that is never a child is the document root.

In [ ]:
SCHEMAS = {
    "elements": pa.schema([
        pa.field("element_id", pa.string(), nullable=False), pa.field("arxiv_id", pa.string(), nullable=False),
        pa.field("kind", pa.dictionary(pa.int32(), pa.string())), pa.field("content", pa.string())]),
    "hierarchy": pa.schema([
        pa.field("arxiv_id", pa.string(), nullable=False), pa.field("version", pa.int32()),
        pa.field("element_id", pa.string(), nullable=False), pa.field("parent_id", pa.string(), nullable=False)]),
    "bibtex": pa.schema([
        pa.field("arxiv_id", pa.string(), nullable=False), pa.field("citation_key", pa.string(), nullable=False),
        pa.field("entry_type", pa.dictionary(pa.int32(), pa.string())), pa.field("title", pa.string()),
        pa.field("authors", pa.list_(pa.string())), pa.field("year", pa.int32()), pa.field("doi", pa.string()),
        pa.field("journal", pa.string()), pa.field("url", pa.string())]),
}


def check_batch(elements, hierarchy):
    """Broken invariants of a batch of papers (columns of elements and hierarchy)."""
    problems = []
    ids = elements["element_id"]
    known = set(ids)
    if len(known) != len(ids):
        problems.append(f"{len(ids) - len(known)} duplicate element_id")
    parents, children = defaultdict(set), defaultdict(set)
    for arxiv_id, version, element_id, parent_id in zip(*(hierarchy[c] for c in HIERARCHY_COLUMNS)):
        if element_id not in known or parent_id is None or parent_id not in known:
            problems.append(f"{arxiv_id} v{version}: edge {element_id} -> {parent_id} outside elements")
        parents[(arxiv_id, version)].add(parent_id)
        children[(arxiv_id, version)].add(element_id)
    for (arxiv_id, version), parent_ids in parents.items():
        roots = parent_ids - children[(arxiv_id, version)]
        if roots != {to_folder_id(arxiv_id) + "#doc"}:
            problems.append(f"{arxiv_id} v{version}: roots {sorted(roots)}")
    return problems


def merge_tables(folder_ids, out_dir=M2_DATA, batch=500, threads=16):
    """The three Parquet tables of the given papers; returns ({table: rows}, problems, bytes written)."""
    local = Path(tempfile.mkdtemp(prefix="m2-tables-"))
    writers = {name: pq.ParquetWriter(local / f"{name}.parquet", schema, compression="zstd")
               for name, schema in SCHEMAS.items()}
    rows, problems = Counter(), []
    bar = tqdm(total=len(folder_ids), unit="paper", desc="Tables", dynamic_ncols=True) if tqdm else None
    try:
        with ThreadPoolExecutor(threads) as readers:
            for start in range(0, len(folder_ids), batch):
                chunk = folder_ids[start:start + batch]
                docs = [d for d in readers.map(lambda f: read_json(DATA_DIR / f / "parsed.json"), chunk) if d]
                columns = {}
                for name, schema in SCHEMAS.items():
                    columns[name] = {field: [v for d in docs for v in d[name][field]] for field in schema.names}
                    writers[name].write_table(pa.Table.from_pydict(columns[name], schema=schema))
                    rows[name] += len(columns[name][schema.names[0]])
                problems += check_batch(columns["elements"], columns["hierarchy"])
                if bar is not None:
                    bar.update(len(chunk))
    finally:
        for writer in writers.values():
            writer.close()
        if bar is not None:
            bar.close()
    out_dir.mkdir(parents=True, exist_ok=True)
    written = 0
    for name in SCHEMAS:
        target = out_dir / f"{name}.parquet"
        shutil.copyfile(local / f"{name}.parquet", target.with_suffix(".parquet.tmp"))
        os.replace(target.with_suffix(".parquet.tmp"), target)
        written += target.stat().st_size
    shutil.rmtree(local, ignore_errors=True)
    return dict(rows), problems, written


done = read_done_index()
in_run = [f for f in PAPERS if f in done]
table_rows, table_problems, parquet_bytes = {}, [], 0
if MERGE_TABLES and in_run:
    table_rows, table_problems, parquet_bytes = merge_tables(in_run)
    expected = {t: sum(done[f]["rows"][t] for f in in_run) for t in SCHEMAS}
    print(f"Tables in {M2_DATA}: " + ", ".join(f"{t} {n:,} rows" for t, n in table_rows.items())
          + f" | {parquet_bytes / MB:,.1f} MB")
    if table_rows != expected:
        table_problems.append(f"row counts {table_rows} differ from the parsed papers {expected}")

## 4. Checks, statistics and benchmark

**Checks** (printed, and `AssertionError` if one fails): the invariants of section 3; `check_paper` on every
paper (well-formed IDs, unique, parents present, one root per version); every version of every paper of the
run is in `logs/m2_parse_status.csv`.

**`logs/m2_stats.json`** (for the report, all sessions; papers of this run):
- parse success rate: `ok` versions over the versions with a `.tex` file, and over all versions;
- version coverage per paper: versions parsed / versions in `metadata.json`, as a distribution (`"2/3": n`)
  and summary counts;
- elements before dedup (every node of every version) and after (rows of `elements`), per `kind`;
- BibTeX entries before dedup (summed over versions) and after (rows of `bibtex`);
- distributions of `bib_source` and `status`.

**`logs/m2_benchmark.json`** (all sessions): `start`, `end`, `wall_seconds` (sum of the sessions),
`papers`, `versions`, `avg_seconds_per_paper` (wall time / papers), `peak_ram_mb` (max), `avg_ram_mb`
(time-weighted), `output_bytes` (`parsed.json` files + Parquet tables), `ram_timeseries` (at most 5,000
`[seconds, MB]` points), `workers`, `colab_cpu`.

In [ ]:
def status_rows():
    """{(arxiv_id, version): last row} of logs/m2_parse_status.csv."""
    rows = {}
    if STATUS_LOG.exists():
        with open(STATUS_LOG, newline="", encoding="utf-8") as stream:
            for row in csv.DictReader(stream):
                try:
                    rows[(row["arxiv_id"], int(row["version"]))] = row
                except (KeyError, TypeError, ValueError):
                    continue
    return rows


def compute_stats(papers, done, logged):
    versions, coverage, nodes, elements, bib_before, bib_after = [], Counter(), Counter(), Counter(), 0, 0
    for folder_id, n_versions in papers.items():
        arxiv_id = folder_id.replace("-", ".", 1)
        summary = done.get(folder_id)
        if summary:
            records = summary["versions"]
            nodes.update(summary["nodes_by_kind"])
            elements.update(summary["elements_by_kind"])
            bib_before += summary["bib_before"]
            bib_after += summary["bib_after"]
        else:   # not parsed (the paper failed): its versions as logged, else as errors
            records = [logged.get((arxiv_id, v)) or {"status": "error", "bib_source": ""} for v in range(1, n_versions + 1)]
        versions += records
        coverage[f"{sum(r['status'] == 'ok' for r in records)}/{n_versions}"] += 1
    status = Counter(r["status"] for r in versions)
    with_tex = sum(status[s] for s in ("ok", "no_root", "error"))
    share = lambda part, whole: round(100 * part / whole, 2) if whole else None   # noqa: E731
    full = sum(n for key, n in coverage.items() if key.split("/")[0] == key.split("/")[1])
    none = sum(n for key, n in coverage.items() if key.startswith("0/"))
    return {
        "computed_at": to_iso(datetime.now(timezone.utc)), "parser_version": PARSER_VERSION,
        "implementation": IMPLEMENTATION,
        "papers": len(papers), "papers_parsed": sum(f in done for f in papers),
        "versions": len(versions), "versions_with_tex": with_tex,
        "parse_success_rate_of_tex_versions_pct": share(status["ok"], with_tex),
        "parse_success_rate_of_all_versions_pct": share(status["ok"], len(versions)),
        "version_coverage": {
            "papers_full": full, "papers_partial": len(papers) - full - none, "papers_none": none,
            "mean_pct": round(statistics.mean(100 * int(k.split("/")[0]) / int(k.split("/")[1])
                                              for k, n in coverage.items() for _ in range(n)), 2) if coverage else None,
            "distribution": dict(sorted(coverage.items(), key=lambda kv: (int(kv[0].split("/")[1]), kv[0]))),
        },
        "elements_before_dedup": sum(nodes.values()), "elements_after_dedup": sum(elements.values()),
        "elements_before_dedup_by_kind": dict(nodes.most_common()),
        "elements_after_dedup_by_kind": dict(elements.most_common()),
        "bib_entries_before_dedup": bib_before, "bib_entries_after_dedup": bib_after,
        "status": dict(status.most_common()),
        "bib_source_of_tex_versions": dict(Counter(r["bib_source"] for r in versions if r["status"] != "empty_source").most_common()),
        "empty_source_reasons": dict(Counter(logged.get((p.replace("-", ".", 1), v), {}).get("error", "")
                                             for p, n in papers.items() for v in range(1, n + 1)
                                             if (logged.get((p.replace("-", ".", 1), v)) or {}).get("status") == "empty_source").most_common()),
    }


def load_m2_sessions():
    records = [read_json(p) for p in M2_SESSIONS.glob("*.json")] if M2_SESSIONS.is_dir() else []
    return sorted((r for r in records if r), key=lambda r: r["start"])


def compute_benchmark(papers, done, output_bytes):
    sessions = load_m2_sessions()
    wall = sum(s["wall_seconds"] for s in sessions)
    parsed = [done[f] for f in papers if f in done]
    series = []
    if M2_RAM_LOG.exists():
        with open(M2_RAM_LOG, newline="", encoding="utf-8") as stream:
            samples = list(csv.DictReader(stream))
        if samples:
            first = datetime.fromisoformat(samples[0]["utc_time"])
            step = max(1, len(samples) // 5000)
            series = [[round((datetime.fromisoformat(s["utc_time"]) - first).total_seconds(), 1), float(s["rss_mb"])]
                      for s in samples[::step]]
    return {
        "start": sessions[0]["start"] if sessions else None, "end": sessions[-1]["end"] if sessions else None,
        "wall_seconds": round(wall, 1), "sessions": len(sessions),
        "papers": len(parsed), "versions": sum(len(p["versions"]) for p in parsed),
        "avg_seconds_per_paper": round(wall / len(parsed), 3) if parsed else None,
        "median_paper_seconds_in_worker": round(statistics.median(p["seconds"] for p in parsed), 3) if parsed else None,
        "peak_ram_mb": max((s["ram_peak_mb"] for s in sessions), default=None),
        "avg_ram_mb": round(sum((s["ram_avg_mb"] or 0) * s["wall_seconds"] for s in sessions) / wall, 2) if wall else None,
        "output_bytes": output_bytes,
        "ram_timeseries": series,
        "workers": WORKERS, "colab_cpu": cpu_info(), "parser_version": PARSER_VERSION,
    }


done = read_done_index()
logged = status_rows()

# Checks
paper_problems = {f: done[f]["problems"] for f in PAPERS if f in done and done[f].get("problems")}
missing_rows = [(f, v) for f, n in PAPERS.items() for v in range(1, n + 1)
                if (f.replace("-", ".", 1), v) not in logged]
failed_papers = [f for f in PAPERS if f not in done]
print(f"Checks: {len(paper_problems)} papers with broken invariants, {len(table_problems)} table problems, "
      f"{len(missing_rows)} versions missing from {STATUS_LOG.name}, {len(failed_papers)} papers not parsed")
for folder_id, problems in list(paper_problems.items())[:5]:
    print("  ", folder_id, problems[:3])
for problem in table_problems[:5]:
    print("  ", problem)

# Statistics
stats = compute_stats(PAPERS, done, logged)
write_json(M2_STATS, stats)

# Benchmark: this session's final record, then every session
monitor.stop()
parsed_bytes = sum(done[f]["bytes"] for f in PAPERS if f in done)
final = session.finish({"output_bytes": parsed_bytes + parquet_bytes, "parquet_bytes": parquet_bytes})
M2_SESSIONS_LOG.write_text("".join(json.dumps(s) + "\n" for s in load_m2_sessions()), encoding="utf-8")
benchmark = compute_benchmark(PAPERS, done, parsed_bytes + parquet_bytes)
write_json(M2_BENCHMARK, benchmark)

b, s = benchmark, stats
lines = [
    ("Papers parsed / in this run", f"{s['papers_parsed']:,} / {s['papers']:,}"),
    ("Versions: ok / empty_source / no_root / error", " / ".join(f"{s['status'].get(k, 0):,}" for k in VERSION_STATUSES)),
    ("Parse success rate: of versions with .tex / of all", f"{s['parse_success_rate_of_tex_versions_pct']}% / "
                                                          f"{s['parse_success_rate_of_all_versions_pct']}%"),
    ("Papers with every version parsed / some / none", f"{s['version_coverage']['papers_full']:,} / "
                                                       f"{s['version_coverage']['papers_partial']:,} / {s['version_coverage']['papers_none']:,}"),
    ("Elements before -> after dedup", f"{s['elements_before_dedup']:,} -> {s['elements_after_dedup']:,}"),
    ("BibTeX entries before -> after dedup", f"{s['bib_entries_before_dedup']:,} -> {s['bib_entries_after_dedup']:,}"),
    ("bib_source (versions with .tex)", ", ".join(f"{k} {v:,}" for k, v in s["bib_source_of_tex_versions"].items())),
    ("Wall time (all sessions) / sessions", f"{b['wall_seconds'] / 60:,.1f} min / {b['sessions']}"),
    ("Average time per paper (wall / papers)", f"{b['avg_seconds_per_paper']} s with {WORKERS} workers"),
    ("Peak RAM / average RAM (with workers)", f"{b['peak_ram_mb']} MB / {b['avg_ram_mb']} MB"),
    ("Output: parsed.json + Parquet", f"{parsed_bytes / MB:,.1f} MB + {parquet_bytes / MB:,.1f} MB"),
    ("Estimate for all papers", f"{b['avg_seconds_per_paper'] * len(ALL_PAPERS) / 3600:,.1f} h for {len(ALL_PAPERS):,} papers"
                                if b["avg_seconds_per_paper"] else "-"),
]
width = max(len(name) for name, _ in lines)
print("\n" + "\n".join(f"{name:<{width}}  {value}" for name, value in lines))
print(f"\nSaved {M2_STATS.name}, {M2_BENCHMARK.name}, {M2_SESSIONS_LOG.name}, {M2_RAM_LOG.name}, {STATUS_LOG.name}")

if IN_COLAB and FLUSH_DRIVE_AT_END:
    from google.colab import drive
    drive.flush_and_unmount()   # Drive writes are asynchronous: make sure every file reached Drive
    print("Google Drive flushed and unmounted: every file of this session is saved. Run all mounts it again.")
assert not (paper_problems or table_problems or missing_rows), "M2 checks failed (see above)"